In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os 
import sys
from pyspark.sql import DataFrame
from typing import List
from pyspark.sql import Window
from delta.tables import DeltaTable


In [0]:
class transformations:
    def dedup(self, df: DataFrame, dedup_cols: List[str], cdc: str):
        df = df.withColumn("dedupKey", concat(*dedup_cols))
        df = df.withColumn("dedupRank", row_number().over(Window.partitionBy("dedupKey").orderBy(col(cdc).desc())))
        df = df.filter(col("dedupRank") == 1)
        df = df.drop("dedupKey", "dedupRank")
        return df
    
    def process_timestamp(self,df):
        df=df.withColumn("process_timestamp",current_timestamp())
        return df
    
    def upsert(self,df,key_cols,table,cdc):
        merge_condition=" AND ".join([f"src.{i}=tgt.{i}" for i in key_cols])
        dlt_obj= DeltaTable.forName(spark,f"dbtsparkproject.silver.{table}")
        dlt_obj.alias("tgt").merge(df.alias("src"),merge_condition)\
                            .whenMatchedUpdateAll(condition=f"src.{cdc}>=tgt.{cdc}")\
                            .whenNotMatchedInsertAll()\
                            .execute()
        return 1                    

In [0]:
df_cust=spark.read.table("dbtsparkproject.bronze.customers")
df_driver=spark.read.table("dbtsparkproject.bronze.drivers")
df_trips=spark.read.table("dbtsparkproject.bronze.trips")
df_locations=spark.read.table("dbtsparkproject.bronze.locations")
df_vehicles=spark.read.table("dbtsparkproject.bronze.vehicles")
df_payments=spark.read.table("dbtsparkproject.bronze.payments")


In [0]:
display(df_driver)

In [0]:
df_cust=df_cust.withColumn("domain",split(col("email"),"@")[1])\
    .withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name")))\
        .withColumn("phone_number",regexp_replace(col("phone_number"),r"[^0-9]",""))
      

In [0]:
current_dir=os.getcwd()
sys.path.append(current_dir)

Customers

In [0]:
cust_obj=transformations()
cust_df_trns=cust_obj.dedup(df_cust,['customer_id'],'last_updated_timestamp')

cust_df_trns=cust_obj.process_timestamp(cust_df_trns)




In [0]:
if not spark.catalog.tableExists("dbtsparkproject.silver.customers"):
    cust_df_trns.write.format("delta").mode("append").saveAsTable("dbtsparkproject.silver.customers")

else:
    cust_obj.upsert(cust_df_trns,['customer_id'],'customers','last_updated_timestamp')


In [0]:
%sql
select count(*) from dbtsparkproject.silver.customers

In [0]:
df_driver=df_driver.withColumn("full_name",concat_ws(" ",col("first_name"),col("last_name")))
df_driver=df_driver.withColumn("phone_number",regexp_replace(col("phone_number"),r"[^0-9]",""))
display(df_driver)


In [0]:
driver_obj=transformations()
driver_df_trns=driver_obj.dedup(df_driver,['driver_id'],'last_updated_timestamp')

driver_df_trns=driver_obj.process_timestamp(driver_df_trns)


In [0]:
if not spark.catalog.tableExists("dbtsparkproject.silver.drivers"):
    driver_df_trns.write.format("delta").mode("append").saveAsTable("dbtsparkproject.silver.drivers")

else:
    driver_obj.upsert(driver_df_trns,['driver_id'],'drivers','last_updated_timestamp')

In [0]:
%sql
select count(*) from dbtsparkproject.silver.customers

In [0]:
display(df_locations)

In [0]:
loc_obj=transformations()
loc_df_trns=loc_obj.dedup(df_locations,['location_id'],'last_updated_timestamp')

loc_df_trns=loc_obj.process_timestamp(loc_df_trns)

In [0]:
if not spark.catalog.tableExists("dbtsparkproject.silver.locations"):
    loc_df_trns.write.format("delta").mode("append").saveAsTable("dbtsparkproject.silver.locations")

else:
    loc_obj.upsert(loc_df_trns,['location_id'],'locations','last_updated_timestamp')


In [0]:
%sql
select count(*) from dbtsparkproject.silver.locations

In [0]:
display(df_payments)

In [0]:
df_payments = df_payments.withColumn("Online_payment_status",
    when(
        (col("payment_method") == "Card") & (col("payment_status") == "Success"), 
        "Online-Success"
    )
    .when(
        (col("payment_method") == "Card") & (col("payment_status") == "Failed"), 
        "Online-Failed"
    )
    .otherwise("Offline")
)

display(df_payments)
                                   
                                   
                                   
                                   
                                   
                                   

In [0]:
pay_obj=transformations()
pay_df_trns=pay_obj.dedup(df_payments,['payment_id'],'last_updated_timestamp')

pay_df_trns=pay_obj.process_timestamp(pay_df_trns)



In [0]:
if not spark.catalog.tableExists("dbtsparkproject.silver.payments"):
    pay_df_trns.write.format("delta").mode("append").saveAsTable("dbtsparkproject.silver.payments")

else:
    pay_obj.upsert(pay_df_trns,['payment_id'],'payments','last_updated_timestamp')

In [0]:
%sql
select count(*) from dbtsparkproject.silver.payments

In [0]:
display(df_vehicles)

In [0]:
df_vehicles=df_vehicles.withColumn("make",upper(col("make")))
veh_obj=transformations()
veh_df_trns=veh_obj.dedup(df_vehicles,['vehicle_id'],'last_updated_timestamp')
veh_df_trns=veh_obj.process_timestamp(veh_df_trns)



In [0]:
if not spark.catalog.tableExists("dbtsparkproject.silver.vehicles"):
    veh_df_trns.write.format("delta").mode("append").saveAsTable("dbtsparkproject.silver.vehicles")

else:
    veh_obj.upsert(veh_df_trns,['vehicle_id'],'vehicles','last_updated_timestamp')
                   

# Trips


In [0]:
display(df_trips)

In [0]:
trip_obj=transformations()
trip_df_trns=trip_obj.dedup(df_trips,['trip_id'],'last_updated_timestamp')
trip_df_trns=trip_obj.process_timestamp(trip_df_trns)

In [0]:
if not spark.catalog.tableExists("dbtsparkproject.silver.trips"):
    trip_df_trns.write.format("delta").mode("append").saveAsTable("dbtsparkproject.silver.trips")

else:    
    trip_obj.upsert(trip_df_trns,['trip_id'],'trips','last_updated_timestamp')

In [0]:
%sql
select count(*) from dbtsparkproject.silver.trips